# Train the MLBB tactical networks (free, in Google Colab)

**Nothing to set.** Upload your CSV files, run every cell (*Runtime → Run all*), download one file: `mlbb_policies.zip`.
Put it in **Download/MLBB-Research** - the app unpacks it and loads the right network by itself.

What it makes, from your recorded play **and** the live matches you watched, together:
- **one network for all heroes** - it is also told each hero's type (role, melee / ranged, starts fights or protects,
  early- or late-game), so it plays a tank differently from a marksman, even heroes with few recordings;
- **one network per hero** you recorded enough of (1500+ moments) - those play best.

While you play, the app uses your hero's own network if there is one, else the all-heroes one. Moments of the 6 s before
a death count less (they led to dying). Re-train every few matches: it gets better with more play.

**Before you start:** app → Learn page → *Export my recorded play (CSV)* and *Export live matches (CSV)*
(files in Downloads/MLBB-Research). *Runtime → Change runtime type*: CPU is enough.

In [ ]:
# TensorFlow and pandas are already installed in Colab
import numpy as np, pandas as pd, tensorflow as tf, math, json, io
print("TensorFlow", tf.__version__)

## 1. Load your files
Easiest: put all your `mlbb-my-play-*.csv` and `mlbb-live-*.csv` files in **My Drive/MLBB-Research** (Google Drive app:
upload them there) - the cell below takes them from there by itself. Otherwise it asks you to pick them on this device.

In [ ]:
# Your files: taken from Google Drive (My Drive/MLBB-Research) when they are there - nothing to upload.
# If that folder has none, you pick them on this device instead. (Google asks once to allow Drive access.)
import glob, os
from google.colab import files
DRIVE_DIR = "/content/drive/MyDrive/MLBB-Research"
paths = []
try:
    from google.colab import drive
    drive.mount("/content/drive")
    for pat in ["mlbb-my-play-*.csv", "mlbb-live-*.csv"]:
        paths += sorted(glob.glob(f"{DRIVE_DIR}/{pat}"))
    paths = sorted(set(paths))
except Exception as e:
    print("Google Drive not used:", e)
if paths:
    print(f"Using {len(paths)} CSV files from Google Drive (My Drive/MLBB-Research):", ", ".join(os.path.basename(p) for p in paths))
else:
    print("No CSV files in My Drive/MLBB-Research - pick them on this device:")
    up = files.upload()
    paths = [os.path.abspath(k) for k in up]
frames = [pd.read_csv(f) for f in paths if f.endswith(".csv")]
df = pd.concat(frames, ignore_index=True)     # my-play and live files mix fine
print(len(df), "rows from", len(frames), "files")

## 2. Train everything
Takes a few minutes. For each network it prints how good it is on your most recent play (which it did not train on):
the joystick number with every move counted equally (40%+ = the app uses its joystick choices) and the buttons good
enough to be used. Parts that are not good enough yet are simply not used by the app.

In [ ]:
# Nothing to set: this trains ONE network for all your heroes (it is told each hero's type) and
# ONE network per hero you recorded enough of, from your play AND the live matches together,
# then packs them all into mlbb_policies.zip for the app.
# (Advanced, normally left alone:)
LIVE_WEIGHT = 1.0       # live-match moments count like yours
DIED_WEIGHT = 0.3       # moments of the 6 s before a death count less (they led to dying)
WATCHED_WEIGHT = 0      # heroes you watched while YOU were dead: left out
MIN_HERO_MOMENTS = 1500 # a hero needs this many moments for its own network (else the all-heroes one plays it)
EPOCHS = 40
TYPE_SIZE = 11          # hero-type numbers (role x6, ranged, starts fights, protects, late game, early game)

import zipfile, re, os
tf.get_logger().setLevel("ERROR")

def parse(col):
    out = []
    for v in df[col].fillna("").astype(str):
        out.append([float(x) for x in v.split(",")] if v.strip() else None)
    return out

f1 = parse("features")
f2 = parse("features2") if "features2" in df.columns else [None] * len(df)
n40 = sum(1 for v in f2 if v is not None and len(v) == 40)
n32 = sum(1 for v in f2 if v is not None and len(v) in (32, 40))
N = 40 if n40 >= 1500 else 32 if n32 >= 1500 else 17          # richest situation with enough rows
use2 = N > 17
df["source"] = df["source"].fillna("me") if "source" in df.columns else "me"
df["died"] = df["died"].fillna(0).astype(int) if "died" in df.columns else 0
has_type = "hero_type" in df.columns
htype = parse("hero_type") if has_type else [None] * len(df)
if not has_type:
    print("NOTE: these CSV files have no hero types (exported before app v36.9). The all-heroes network is made "
          "without them - export again from the app for the better one.")
print(f"{len(df)} rows; situation: {N} numbers; moments: {(df['source'] == 'me').sum()} yours, "
      f"{(df['source'] == 'live').sum()} from live matches, {(df['died'] == 1).sum()} before a death")

# ---- every usable moment once ----
X, XT, MOVE, CASTS, W, TS, HEROES = [], [], [], [], [], [], []
for i, row in df.iterrows():
    hero = str(row["hero"]).strip()
    watched = hero.startswith("~")
    if not hero or (watched and WATCHED_WEIGHT <= 0): continue
    feat = f2[i] if use2 else f1[i]
    if feat is None or len(feat) < N: continue
    mx, my = float(row["move_x"]), float(row["move_y"])
    move = 8 if math.hypot(mx, my) < 0.4 else int(round(math.atan2(my, mx) / (math.pi / 4))) % 8
    c = int(row["casts"])
    wgt = WATCHED_WEIGHT if watched else 1.0
    if row["source"] == "live": wgt *= LIVE_WEIGHT
    if int(row["died"]) == 1: wgt *= DIED_WEIGHT
    if wgt <= 0: continue
    t = htype[i] if htype[i] is not None and len(htype[i]) == TYPE_SIZE else [0.0] * TYPE_SIZE
    X.append(feat[:N]); XT.append(t); MOVE.append(move); CASTS.append([(c >> b) & 1 for b in range(6)])
    W.append(wgt); TS.append(int(row["ts"])); HEROES.append(hero)
X = np.array(X, np.float32); XT = np.array(XT, np.float32); MOVE = np.array(MOVE); CASTS = np.array(CASTS, np.float32)
W = np.array(W, np.float32); TS = np.array(TS); HEROES = np.array(HEROES)
order = np.argsort(TS, kind="stable")
X, XT, MOVE, CASTS, W, TS, HEROES = X[order], XT[order], MOVE[order], CASTS[order], W[order], TS[order], HEROES[order]
# a press counts for its neighbours too (within 0.45 s, same hero): "a little early / late" is not wrong
wide = CASTS.copy()
for d in (-1, 1):
    j = np.arange(len(TS)) + d; ok = (j >= 0) & (j < len(TS)); jj = np.clip(j, 0, len(TS) - 1)
    near = ok & (np.abs(TS[jj] - TS) <= 450) & (HEROES[jj] == HEROES)
    wide[near] = np.maximum(wide[near], CASTS[jj][near])
CASTS = wide
FIGHT = (X[:, 6] > 0.5) | (X[:, 7] > 0)
if N >= 32: FIGHT |= X[:, 20] > 0
own = np.array([not h.startswith("~") for h in HEROES])
counts = pd.Series(HEROES[own]).value_counts()
print("moments per hero:", ", ".join(f"{h} {c}" for h, c in counts.items()))
assert len(X) >= 300, "Too little recorded play: play more matches with screen capture on, then export again."

BTN = ["attack", "skill 1", "skill 2", "ultimate", "spell", "recall"]
KEYS = ["attack", "s1", "s2", "ult", "spell", "recall"]
APP = np.array([0.5, 0.6, 0.6, 0.6, 0.6, 0.85], np.float32)   # the app's own press lines

def train(sel, with_type, hero_label):
    """Trains one network on the moments in `sel`; returns (tflite bytes, info dict, one-line report)."""
    Xs = np.concatenate([X[sel], XT[sel]], 1) if with_type else X[sel]
    M, C, Ws, F = MOVE[sel], CASTS[sel], W[sel], FIGHT[sel]
    cut = int(len(Xs) * 0.85)                          # the most recent 15% checks the result
    n_in = Xs.shape[1]
    inp = tf.keras.Input(shape=(n_in,), name="situation")
    h = tf.keras.layers.Dense(128, activation="relu")(inp)
    h = tf.keras.layers.Dropout(0.1)(h)
    h = tf.keras.layers.Dense(64, activation="relu")(h)
    move_out = tf.keras.layers.Dense(9, activation="softmax", name="move")(h)
    cast_out = tf.keras.layers.Dense(6, activation="sigmoid", name="casts")(h)
    model = tf.keras.Model(inp, [move_out, cast_out])
    # lopsided recordings: every move counts as much as the others, a press as much as the no-presses
    Ft = F[:cut]; cnt = np.bincount(M[:cut][Ft], minlength=9); present = cnt > 0
    cw = np.clip(np.where(present, Ft.sum() / (max(1, present.sum()) * np.maximum(cnt, 1)), 0.0), 0.0, 15.0)
    sw_move = Ws[:cut] * cw[M[:cut]] * Ft
    pos = C[:cut].sum(0)
    POS_W = tf.constant(np.clip((len(C[:cut]) - pos) / np.maximum(pos, 1), 1.0, 30.0), tf.float32)
    def weighted_bce(y, p):
        p = tf.clip_by_value(p, 1e-6, 1 - 1e-6)
        return -tf.reduce_mean(POS_W * y * tf.math.log(p) + (1 - y) * tf.math.log(1 - p), axis=-1)
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss=["sparse_categorical_crossentropy", weighted_bce])
    model.fit(Xs[:cut], [M[:cut], C[:cut]], sample_weight=[sw_move, Ws[:cut]],
              validation_data=(Xs[cut:], [M[cut:], C[cut:]]), epochs=EPOCHS, batch_size=128, verbose=0,
              callbacks=[tf.keras.callbacks.EarlyStopping(patience=6, restore_best_weights=True)])
    # how good, on recent play it did not train on
    pm_all, pc = model.predict(Xs[cut:], verbose=0)
    FV = F[cut:]; pm = pm_all[FV]; yv = M[cut:][FV]
    per = [((pm.argmax(1) == k) & (yv == k)).sum() / (yv == k).sum() for k in range(9) if (yv == k).sum() > 0]
    bal = float(np.mean(per)) if per else 0.0
    THR, QUALITY = [], {}
    for b in range(6):
        y = C[cut:, b] > 0
        if y.sum() < 5: THR.append(float(APP[b])); continue
        best_t, best_f = float(APP[b]), -1.0
        for t in np.arange(0.05, 0.96, 0.05):
            p = pc[:, b] > t; tp = (p & y).sum(); f = 2 * tp / max(1, p.sum() + y.sum())
            if f > best_f: best_f, best_t = f, float(t)
        THR.append(best_t); p = pc[:, b] > best_t
        QUALITY[KEYS[b]] = {"checked": int(y.sum()), "right": float((p & y).sum() / max(1, p.sum())), "found": float((p & y).sum() / y.sum())}
    # export with the press lines built in (so they match the app's)
    logit = lambda v: np.log(v / (1 - v))
    SHIFT = tf.constant(logit(APP) - logit(np.clip(np.array(THR, np.float32), 0.05, 0.95)), tf.float32)
    class Calibrate(tf.keras.layers.Layer):
        def call(self, p):
            p = tf.clip_by_value(p, 1e-6, 1 - 1e-6)
            return tf.sigmoid(tf.math.log(p / (1 - p)) + SHIFT)
    export = tf.keras.Model(inp, tf.keras.layers.Concatenate(name="policy")([move_out, Calibrate()(cast_out)]))
    import contextlib
    with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
        tfl = tf.lite.TFLiteConverter.from_keras_model(export).convert()     # (its long printout hidden)
    seen = np.bincount(yv, minlength=9)
    fewest = int(seen[seen > 0].min()) if (seen > 0).any() else 0
    info = {"hero": hero_label, "hero_type_inputs": bool(with_type), "inputs": int(n_in), "outputs": 15, "examples": int(len(Xs)),
            "move_balanced": bal, "move_checked": int(len(yv)), "move_fewest": fewest, "buttons": QUALITY, "press_lines": THR,
            "live_weight": LIVE_WEIGHT, "died_weight": DIED_WEIGHT}
    joy = "used" if bal >= 0.40 and len(yv) >= 300 and fewest >= 15 else "not yet"
    btns = [k for k, q in QUALITY.items() if q["checked"] >= 20 and q["right"] >= 0.4]
    report = f"{len(Xs)} moments | joystick {bal:.0%} ({joy}) | buttons used: {', '.join(btns) or 'none yet'}"
    return tfl, info, report

def safe(name): return re.sub(r"[^A-Za-z0-9]+", "_", name).strip("_") or "hero"

out = {}
print("\nall heroes (with hero types) ...", end=" ", flush=True)
tfl, info, rep = train(own | (WATCHED_WEIGHT > 0), has_type, "")
out["all"] = (tfl, info); print(rep)
for hero, n in counts.items():
    sel = own & (HEROES == hero)
    if n < MIN_HERO_MOMENTS or FIGHT[sel].sum() < 300:
        print(f"{hero}: {n} moments - the all-heroes network plays it (needs {MIN_HERO_MOMENTS}+ for its own)")
        continue
    print(f"{hero} ...", end=" ", flush=True)
    tfl, info, rep = train(sel, False, hero)
    out[safe(hero)] = (tfl, info); print(rep)

with zipfile.ZipFile("/content/mlbb_policies.zip", "w") as z:
    for k, (tfl, info) in out.items():
        z.writestr(f"policy_{k}.tflite", tfl)
        z.writestr(f"policy_info_{k}.json", json.dumps(info))
print(f"\nmlbb_policies.zip: {len(out)} networks ({', '.join(out)}) - put it in Download/MLBB-Research")


## 3. Download

In [ ]:
import shutil
# A. Direct download (works best on a computer)
from google.colab import files
files.download("/content/mlbb_policies.zip")
try:   # also keep a copy in My Drive/MLBB-Research when Google Drive is connected
    if os.path.isdir("/content/drive/MyDrive"):
        os.makedirs("/content/drive/MyDrive/MLBB-Research", exist_ok=True)
        shutil.copy("/content/mlbb_policies.zip", "/content/drive/MyDrive/MLBB-Research/")
        print("Also saved to Google Drive > My Drive/MLBB-Research")
except Exception as e: print("(not copied to Drive:", e, ")")

In [ ]:
# B. Save to Google Drive (most reliable on a phone)
from google.colab import drive; import shutil, os
drive.mount("/content/drive"); os.makedirs("/content/drive/MyDrive/MLBB-Research", exist_ok=True)
shutil.copy("/content/mlbb_policies.zip", "/content/drive/MyDrive/MLBB-Research/mlbb_policies.zip")
print("Saved to Google Drive > My Drive/MLBB-Research: mlbb_policies.zip")

## 4. Put it in the app
1. Put `mlbb_policies.zip` in **Download/MLBB-Research** on the phone (replace the old one). The app unpacks it by itself
   (Model page: *Scan the folder now*, or restart the app).
2. The Model page lists the networks on the phone; while playing, the right one is loaded when your hero is known.
3. Only the parts each network got good at are used; the safety rules (retreat, tower circle, recall) always come first.